## 1. Environment Setup and Library Imports


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# -------------------------------------------------------------------------
# 1. Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model

# -------------------------------------------------------------------------
# 2. Hardware and Training Hyperparameters (HC Thickness Prediction - Solo AZ)
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Target material & acceleration modality
target_material = "HC"
sensor_modality = "AZ"
INPUT_CHANNELS = 1

# Classification setup: 5 target sheet thicknesses
thickness_list = ["T185", "T188", "T191", "T194", "T197"]
thickness_to_id = {t: idx for idx, t in enumerate(thickness_list)}
NUM_CLASSES_THICKNESS = len(thickness_list)  # 5 classes
NUM_EPOCHS = 35

# Sample sizes to evaluate sample efficiency (stratified across the 5 classes)
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{s}" for s in sample_sizes]

# Random seeds (10 seeds matching standard benchmark)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# 3. Path Verification for HC Material across all 5 Thicknesses (AZ Only)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 1: Verifying HC Paths for Thickness Prediction (Solo AZ)...")
print("==================================================================")

all_paths_ok = True
hc_az_paths = {}

for t in thickness_list:
    az_p = DATA_DIR / target_material / t / sensor_modality
    exists_az = az_p.exists()
    
    if not exists_az:
        all_paths_ok = False
        
    print(f"[{target_material} {t} {sensor_modality}] {az_p} -> {'FOUND' if exists_az else 'MISSING'}")
    hc_az_paths[t] = str(az_p)

print(f"\nPath status: {'ALL PATHS OK' if all_paths_ok else 'WARNING: SOME PATHS ARE MISSING'}")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Target Material: {target_material}")
print(f"Modality: {sensor_modality} (Input channels: {INPUT_CHANNELS})")
print(f"Thickness Classes: {thickness_to_id} (Total: {NUM_CLASSES_THICKNESS})")
print(f"Evaluation Subsets: {subset_names}")
print(f"Total Random Seeds: {len(seeds)}")

## 2. Data Ingestion & In-Memory Preloading  (AZ Acceleration Sensor)

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from src.pkldataset import PKLDataset

# -------------------------------------------------------------------------
# STEP 2: Load HC Data Across All 5 Thicknesses (AZ Only) & Assign Thickness Labels
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 2: Loading AZ Acceleration Signals for All 5 HC Sheet Thicknesses...")
print("==================================================================")

cpu_cores = os.cpu_count() or 4
num_workers = min(8, cpu_cores)

# Container by thickness: {'inputs': tensor[N, 1, L], 'targets': tensor[N]}
hc_az_tensors = {}

for t_name in thickness_list:
    folder_path = hc_az_paths[t_name]
    t_id = thickness_to_id[t_name]  # 0: T185, 1: T188, 2: T191, 3: T194, 4: T197
    
    # Load raw single-modality acceleration signals (AZ)
    ds = PKLDataset(folder_path=folder_path, modality="single")
    loader = DataLoader(
        ds, 
        batch_size=512, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=False,
        persistent_workers=False
    )
    
    xs = []
    for x, _ in loader:  # Discard the original 10-class wear label
        xs.append(x)
        
    inputs = torch.cat(xs, dim=0)  # Tensor shape: [N, 1, L]
    
    # Assign target label corresponding to current sheet thickness ID
    targets = torch.full((inputs.shape[0],), fill_value=t_id, dtype=torch.long)
    
    hc_az_tensors[t_name] = {
        "inputs": inputs,
        "targets": targets
    }
    print(f"[{t_name}] ID: {t_id} | Cached samples: {inputs.shape[0]} | Tensor shape: {tuple(inputs.shape)}")

print("\nAll HC thickness datasets (AZ) are successfully loaded and cached in RAM.")

## 3. Train/Test Partitioning  - AZ

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset

# -------------------------------------------------------------------------
# STEP 3: Train/Test Partitioning & Stratified Sampling Setup (HC - Solo AZ)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 3: Partitioning AZ Data into Train Pools and Consolidated Test Set...")
print("==================================================================")

TEST_RATIO = 0.20
EVAL_SEED = 42

np.random.seed(EVAL_SEED)

train_pools = {}
test_inputs_list, test_targets_list = [], []

for t_name in thickness_list:
    inputs = hc_az_tensors[t_name]["inputs"]
    targets = hc_az_tensors[t_name]["targets"]
    n_samples = inputs.shape[0]
    
    indices = np.random.permutation(n_samples)
    n_test = int(n_samples * TEST_RATIO)
    
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    
    # Pool available to draw stratified training subsets (train_20 ... train_500)
    train_pools[t_name] = {
        "inputs": inputs[train_idx],
        "targets": targets[train_idx]
    }
    
    test_inputs_list.append(inputs[test_idx])
    test_targets_list.append(targets[test_idx])
    
    print(f"[{t_name}] Train pool: {len(train_idx)} samples | Test set: {len(test_idx)} samples")

# Concatenate all test partitions into a single evaluation tensor on device
X_test = torch.cat(test_inputs_list, dim=0).to(device)
Y_test = torch.cat(test_targets_list, dim=0).to(device)

print(f"\nConsolidated Test Set shape: {tuple(X_test.shape)}")
print(f"Consolidated Test Targets shape: {tuple(Y_test.shape)}")

# -------------------------------------------------------------------------
# Stratified Sampling Helper for Multi-Class Thickness Prediction (5 Classes)
# -------------------------------------------------------------------------
def get_balanced_thickness_subset(train_pools, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset across all 5 thickness classes.
    For example: 20 samples -> 4 samples per thickness class (T185..T197).
    Returns concatenated input and target tensors for the training batch.
    """
    np.random.seed(seed)
    num_classes = len(thickness_list)  # 5 classes
    samples_per_class = total_samples // num_classes
    
    selected_x, selected_y = [], []
    
    for t_name in thickness_list:
        pool_x = train_pools[t_name]["inputs"]
        pool_y = train_pools[t_name]["targets"]
        total_available = pool_x.shape[0]
        
        if total_available < samples_per_class:
            raise ValueError(
                f"Not enough samples in {t_name} pool: "
                f"{total_available} available vs {samples_per_class} requested."
            )
            
        chosen_indices = np.random.choice(total_available, size=samples_per_class, replace=False)
        selected_x.append(pool_x[chosen_indices])
        selected_y.append(pool_y[chosen_indices])
        
    batch_x = torch.cat(selected_x, dim=0)
    batch_y = torch.cat(selected_y, dim=0)
    
    # Shuffle the concatenated balanced batch
    perm = torch.randperm(batch_x.size(0))
    return batch_x[perm], batch_y[perm]

# -------------------------------------------------------------------------
# Results Container Initialization
# -------------------------------------------------------------------------
results = {
    s_name: {
        "overall": [],
        **{t: [] for t in thickness_list}
    }
    for s_name in subset_names
}

print(f"\nStep 3 completed successfully:")
print(f"  Target Classes: {thickness_list} ({NUM_CLASSES_THICKNESS} classes)")
print(f"  Evaluation Subsets: {subset_names}")
print(f"  Random Seeds: {len(seeds)}")

## 4. Training and Evaluation - AZ

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -------------------------------------------------------------------------
# 1. Fast in-memory evaluation helper for thickness prediction
# -------------------------------------------------------------------------
def fast_eval_thickness(model, x_test, y_test, batch_size=512):
    model.eval()
    all_preds = []
    total = y_test.size(0)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = x_test[i:i + batch_size]
            preds = model(x).argmax(dim=1)
            all_preds.append(preds)
            
    all_preds = torch.cat(all_preds, dim=0)
    
    # Overall accuracy across all 5 thickness classes
    overall_acc = 100.0 * (all_preds == y_test).sum().item() / total
    
    # Accuracy broken down by each thickness class
    class_accs = {}
    for t_name, t_id in thickness_to_id.items():
        mask = (y_test == t_id)
        if mask.sum().item() > 0:
            acc = 100.0 * (all_preds[mask] == t_id).sum().item() / mask.sum().item()
            class_accs[t_name] = acc
        else:
            class_accs[t_name] = 0.0
            
    return overall_acc, class_accs

# -------------------------------------------------------------------------
# 2. Reset results container to guarantee clean accumulation
# -------------------------------------------------------------------------
results = {
    s_name: {
        "overall": [],
        **{t: [] for t in thickness_list}
    }
    for s_name in subset_names
}

# -------------------------------------------------------------------------
# 3. Stabilized Training Loop for HC Thickness Prediction (Solo AZ)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 4: Training 1D-CNN (AZ Only) for HC Thickness Prediction (5 Classes)...")
print("==================================================================")

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running HC Thickness Prediction (AZ) for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)
    
    for s_size, s_name in zip(sample_sizes, subset_names):
        # Stratified extraction from HC pools (equal samples per thickness)
        train_x, train_y = get_balanced_thickness_subset(
            train_pools=train_pools,
            total_samples=s_size,
            seed=seed
        )
        
        # Adaptive batch sizing: 32 for small sets, 64 for >= 200 samples
        if s_size < 64:
            curr_batch_size = max(8, s_size // NUM_CLASSES_THICKNESS)
        elif s_size < 200:
            curr_batch_size = 32
        else:
            curr_batch_size = 64
            
        train_loader = DataLoader(
            TensorDataset(train_x, train_y),
            batch_size=curr_batch_size,
            shuffle=True
        )
        
        # 1D-CNN: 1 input channel (AZ), 5 output classes (T185..T197)
        model = get_model(input_channels=INPUT_CHANNELS, num_classes=NUM_CLASSES_THICKNESS).to(device)
        
        # Stabilized optimization parameters
        optimizer = optim.Adam(model.parameters(), lr=7e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.2)
        
        # Train from scratch
        model = train_model(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=NUM_EPOCHS,
            device=device
        )
        
        # Evaluate on the consolidated unseen test set
        overall_acc, class_accs = fast_eval_thickness(
            model=model,
            x_test=X_test,
            y_test=Y_test,
            batch_size=512
        )
        
        # Store metrics
        results[s_name]["overall"].append(overall_acc)
        for t_name in thickness_list:
            results[s_name][t_name].append(class_accs[t_name])
            
        print(f"[Seed {seed}] {s_name} ({s_size} samples) -> Test Overall Acc: {overall_acc:.2f}% | " +
              " | ".join([f"{t}: {class_accs[t]:.1f}%" for t in thickness_list]))

print("\nStep 4 completed successfully. Results stored in 'results'.")

## 5. Summary of Results - AZ

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------------
# STEP 5: Compile Final Summary Table and Export (HC Thickness - Solo AZ)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 5: Compiling Consolidated Results Table (HC Thickness - Solo AZ)...")
print("==================================================================")

summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    
    # Per-class accuracy (recall) for each sheet thickness
    for t_name in thickness_list:
        class_scores = results[s_name][t_name]
        mean_c = np.mean(class_scores)
        std_c = np.std(class_scores)
        row[t_name] = f"{mean_c:.2f} ± {std_c:.2f}"
        
    summary_data.append(row)

# Strict column ordering: Subset -> T185 ... T197
column_order = ["Subset"] + thickness_list
df_thickness_az = pd.DataFrame(summary_data)[column_order]

print(f"\nConsolidated Thickness Prediction Results (HC - Solo AZ) [Mean ± Std over {len(seeds)} seeds]:")
display(df_thickness_az)

# Save to CSV
output_csv = "AZ_thickness_pred_HC_summary.csv"
df_thickness_az.to_csv(output_csv, index=False)
print(f"\nFinal results successfully exported to: {output_csv}")